# FE 11 · Plant sensors vs Reuniwatt forecasts
The PV telemetry table also has the plants' **own sensors** (measured sunlight, temperature, humidity, wind). Reuniwatt **forecasts** the same quantities.
This notebook lines the two up and answers:
1. How close is Reuniwatt to what the sensors measured, and how does that change with lead time?
2. Which input tracks the **target** best: measured sunlight or Reuniwatt's forecast?
3. Where and when is Reuniwatt biased (hour of day, month, specific days)?

**Targets:** PV1 = actual MW · PV2 = `Pv2_AVL` (available power).
**Labels:** PV1 = small plant (east/west panels), PV2 = large plant.

Run cell by cell; each step ends with **✅ What you should see**. Step 5 saves two working tables to `output_schema` so later steps are fast. The last step writes `results/sensor_vs_reuniwatt.txt` next to this notebook: commit and push it to share.

⚠️ Sensor values are **measurements**. In models they can only be used **up to the forecast origin**, never for future targets. Here we compare at the same time to measure quality, which is fine.

## Step 0 · Settings
Default window: Jul–Oct 2026. Everything is correctly labelled in that period, and PV2's panel sensor exists (from Jan 2026). Widen it later (sensors go back to Mar 2023).

In [0]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema (working tables)")
dbutils.widgets.text("start_date", "2026-07-01", "Window start (UTC)")
dbutils.widgets.text("end_date", "2026-10-13", "Window end (UTC)")

In [0]:
%run ./00_fe_common

In [0]:
from datetime import date as _date, timedelta as _td

START = dbutils.widgets.get("start_date").strip()
END = dbutils.widgets.get("end_date").strip()
RUNS_FROM = str(_date.fromisoformat(START) - _td(days=8))  # week-ahead runs issued up to 7 days before the window
print(f"Window {START} → {END} (UTC); reading Reuniwatt runs from {RUNS_FROM}")

# Part A · Plant sensors
## Step 1 · Raw sensor columns: how full are they?

In [0]:
SENSOR_COLS = ["PV1_GHI", "PV1_ArrayRadiance_east", "PV1_ArrayRadiance_west", "PV1_Temp", "PV1_HumidityPer",
               "PV2_GHI", "PV2_ArrayRadiance", "PV2_Outdoor_Temp", "PV2_HumidityPer", "PV2_WindSpeed", "PV2_WindDirection",
               "PV1_MW", "PV2_MW", "Pv2_AVL"]
raw = (
    spark.table(PV_BRONZE).withColumn("ts_gulf", F.col("DateTime").cast("timestamp"))
    .where(F.to_date("ts_gulf").between(F.date_sub(F.lit(START), 1), F.date_add(F.lit(END), 1)))
)
display(raw.agg(F.count("*").alias("rows"), *[F.round(100 * F.avg(F.col(c).isNotNull().cast("int")), 1).alias(c) for c in SENSOR_COLS]))

✅ **What you should see:** % filled per column. Irradiance (GHI, ArrayRadiance) and MW columns ~50% (blank at night), temperature / humidity / wind ~95%+, `Pv2_AVL` ~50%. Any column near 0% is unusable in this window.

## Step 2 · Clean the sensor values
| Type | Rule |
|---|---|
| Irradiance | below −20 → blank; −20 to 0 → 0; above 1,500 → blank |
| Temperature | 0 or below, or above 60 °C → blank (0 means "no reading") |
| Humidity | 0 or below, or above 100% → blank |
| Wind speed | below 0 or above 60 → blank |
| Wind direction | below 0 or above 360° → blank |
| `Pv2_AVL` | below 0 or above 2,000 MW → blank |

In [0]:
NULL = F.lit(None).cast("double")
CLEAN_RULES = {
    "irradiance": (["PV1_GHI", "PV1_ArrayRadiance_east", "PV1_ArrayRadiance_west", "PV2_GHI", "PV2_ArrayRadiance"],
                   lambda v: F.when(v < -20, NULL).when(v < 0, F.lit(0.0)).when(v > 1500, NULL).otherwise(v)),
    "temperature": (["PV1_Temp", "PV2_Outdoor_Temp"], lambda v: F.when((v <= 0) | (v > 60), NULL).otherwise(v)),
    "humidity": (["PV1_HumidityPer", "PV2_HumidityPer"], lambda v: F.when((v <= 0) | (v > 100), NULL).otherwise(v)),
    "wind_speed": (["PV2_WindSpeed"], lambda v: F.when((v < 0) | (v > 60), NULL).otherwise(v)),
    "wind_dir": (["PV2_WindDirection"], lambda v: F.when((v < 0) | (v > 360), NULL).otherwise(v)),
    "available": (["Pv2_AVL"], lambda v: F.when((v < 0) | (v > 2000), NULL).otherwise(v)),
}
clean = raw
for _, (cols, rule) in CLEAN_RULES.items():
    for c in cols:
        clean = clean.withColumn(f"{c}__clean", rule(F.col(c).cast("double")))
display(clean.agg(*[(F.count(c) - F.count(f"{c}__clean")).alias(f"{c}_removed")
                    for cols, _ in CLEAN_RULES.values() for c in cols]))

✅ **What you should see:** values removed per column. A handful is normal. Thousands in one column (e.g. humidity > 100, wind direction > 360) means that sensor needs a closer look.

## Step 3 · One row per plant per 5 minutes, in UTC
- Map columns to plants. PV1: GHI, panel sunlight = average of east and west, temperature, humidity. PV2: GHI, panel sunlight, temperature, humidity, wind.
- Actual MW uses the Bronze column-swap correction.
- **Target:** PV1 = actual MW; PV2 = `Pv2_AVL`.
- Gulf time → UTC, averaged into 5-min periods labelled by their end (same as Reuniwatt), plus the computed sun position.

In [0]:
swap_day = detect_pv_swap_date()


def cc(c):
    return F.col(f"{c}__clean")


per_site = {
    "pv1": dict(m_ghi=cc("PV1_GHI"), m_gti_east=cc("PV1_ArrayRadiance_east"), m_gti_west=cc("PV1_ArrayRadiance_west"),
                m_gti_poa=(cc("PV1_ArrayRadiance_east") + cc("PV1_ArrayRadiance_west")) / 2,
                m_temp_c=cc("PV1_Temp"), m_humidity_pct=cc("PV1_HumidityPer"), m_wind_speed=NULL, m_wind_dir=NULL,
                actual_mw=site_mw("pv1", swap_day), avl_mw=NULL),
    "pv2": dict(m_ghi=cc("PV2_GHI"), m_gti_east=NULL, m_gti_west=NULL, m_gti_poa=cc("PV2_ArrayRadiance"),
                m_temp_c=cc("PV2_Outdoor_Temp"), m_humidity_pct=cc("PV2_HumidityPer"),
                m_wind_speed=cc("PV2_WindSpeed"), m_wind_dir=cc("PV2_WindDirection"),
                actual_mw=site_mw("pv2", swap_day), avl_mw=cc("Pv2_AVL")),
}
MEAS = list(per_site["pv1"].keys())
long = reduce(DataFrame.unionByName, [clean.select(F.lit(s).alias("site"), "ts_gulf", *[e.alias(k) for k, e in exprs.items()])
                                      for s, exprs in per_site.items()])
sens = (
    long.withColumn("period_end", ceil_to(gulf_to_utc("ts_gulf"), STEP_MIN))
    .groupBy("site", "period_end").agg(*[F.avg(k).alias(k) for k in MEAS], F.count("*").alias("n_minutes"))
    .where(F.to_date("period_end").between(F.lit(START), F.lit(END)))
    .withColumn("target_mw", F.when(F.col("site") == "pv2", F.col("avl_mw")).otherwise(F.col("actual_mw")))
)
sens = add_solar(sens, "period_end")
display(sens.groupBy("site").agg(
    F.count("*").alias("periods"),
    *[F.round(100 * F.avg(F.col(k).isNotNull().cast("int")), 1).alias(f"{k}_pct") for k in ["m_ghi", "m_gti_poa", "m_temp_c", "target_mw"]],
    F.round(F.max("target_mw")).alias("target_max"),
))

✅ **What you should see:** 288 periods per day per site. `target_max` around **850–950** for pv1 and **~1,640** for pv2 (the `Pv2_AVL` cap). Irradiance and target ~50% filled (night blank).

## Step 4 · Are the sensors on the same clock?
Average measured GHI by UTC hour next to the computed clear-sky GHI. If the sensor clock is right (Gulf time converted to UTC), both peak around **08:00 UTC**.

In [0]:
display(sens.groupBy("site", F.hour("period_end").alias("utc_hour")).agg(
    F.round(F.avg("m_ghi"), 1).alias("measured_ghi"), F.round(F.avg("clearsky_ghi"), 1).alias("clearsky_ghi"),
    F.round(F.avg("target_mw"), 1).alias("target_mw")).orderBy("site", "utc_hour"))

✅ **What you should see:** `measured_ghi`, `clearsky_ghi` and `target_mw` rise from ~02:00, peak at ~08:00 and fall to 0 by ~14:00 UTC. Measured GHI should be a bit below clear-sky (haze/dust). A ~4-hour shift means the sensor clock differs; stop and flag it.

# Part B · Reuniwatt forecasts
## Step 5 · Read Reuniwatt from Bronze and save both working tables
Uses the shared reader (`reuniwatt_from_bronze`): every intraday run, hourly day-ahead, 3-hourly week-ahead, power in MW. Each forecast row gets a **lead bucket** (how far ahead it was made):

| Horizon | Buckets |
|---|---|
| intraday | 0–1 h, 1–3 h, 3–6 h |
| day-ahead | 0–6 h, 6–12 h, 12–24 h |
| week-ahead | day 1 … day 7 |

Saved as `fe_cmp_sensors_5min` and `fe_cmp_reuniwatt` in `output_schema`. This is the slow step (a few minutes).

In [0]:
lead_h = F.col("lead_min") / 60
bucket = (
    F.when(F.col("horizon") == "intraday", F.when(lead_h <= 1, "0-1h").when(lead_h <= 3, "1-3h").otherwise("3-6h"))
    .when(F.col("horizon") == "dayahead", F.when(lead_h <= 6, "0-6h").when(lead_h <= 12, "6-12h").otherwise("12-24h"))
    .otherwise(F.concat(F.lit("day "), F.ceil(lead_h / 24).cast("int").cast("string")))
)
fc = (
    reuniwatt_from_bronze(RUNS_FROM, END)
    .where(F.to_date("period_end").between(F.lit(START), F.lit(END)))
    .withColumn("lead_bucket", bucket)
    .select("site", "horizon", "reference_time", "period_end", "lead_min", "lead_bucket", "raw_source",
            "ghi", "dni", "dhi", "gti_poa", "temp_c", "humidity_pct", "wind_speed", "wind_dir", "cloud_opacity",
            "power_mw", "power_p10_mw", "power_p90_mw")
)
sens = write(sens, "fe_cmp_sensors_5min", partition_by=("site",))
fc = write(fc, "fe_cmp_reuniwatt", partition_by=("horizon",))
display(fc.groupBy("site", "horizon", "lead_bucket").agg(F.countDistinct("reference_time").alias("runs"), F.count("*").alias("rows"))
        .orderBy("site", "horizon", "lead_bucket"))

✅ **What you should see:** for each site, 3 intraday buckets, 3 day-ahead buckets and 7 week-ahead days, all with rows. The printed unit line should say **kW** (× 0.001).

# Part C · Comparisons
## Step 6 · Reuniwatt sunlight vs measured sunlight
For GHI and panel sunlight (GTI), daytime only (sun more than 5° up), by site × horizon × lead bucket:
- `corr`: how well the shapes match (1 = perfect)
- `bias`: average of forecast − measured (+ = Reuniwatt too high)
- `mae`: average size of the error
- `nmae_pct`: MAE as % of the average measured value

In [0]:
cmp = fc.join(sens, ["site", "period_end"]).where("sun_elevation > 5")


def compare(df: DataFrame, f: str, m: str, label: str, by) -> DataFrame:
    d = df.where(F.col(f).isNotNull() & F.col(m).isNotNull())
    return d.groupBy(*by).agg(
        F.count("*").alias("n"),
        F.round(F.corr(f, m), 3).alias("corr"),
        F.round(F.avg(F.col(f) - F.col(m)), 1).alias("bias"),
        F.round(F.avg(F.abs(F.col(f) - F.col(m))), 1).alias("mae"),
        F.round(100 * F.avg(F.abs(F.col(f) - F.col(m))) / F.when(F.avg(m) > 0, F.avg(m)), 1).alias("nmae_pct"),
    ).withColumn("variable", F.lit(label))


BY = ["site", "horizon", "lead_bucket"]
irr_cmp = compare(cmp, "ghi", "m_ghi", "GHI", BY).unionByName(compare(cmp, "gti_poa", "m_gti_poa", "GTI (panels)", BY)) \
    .orderBy("variable", "site", "horizon", "lead_bucket")
display(irr_cmp)

✅ **What you should see:** `corr` high (**0.9+**) and `nmae_pct` low (~5–15%) for intraday 0–1 h, getting worse with lead: day-ahead, then week-ahead day 7. `bias` tells you whether Reuniwatt runs systematically high or low. A large constant bias is easy to correct with a feature.

## Step 7 · Reuniwatt weather vs measured weather (day-ahead only)
Temperature, humidity and wind speed come only with day-ahead runs. All hours, by site × lead bucket.

In [0]:
da = fc.where("horizon = 'dayahead'").join(sens, ["site", "period_end"])
BYW = ["site", "lead_bucket"]
wx_cmp = reduce(DataFrame.unionByName, [
    compare(da, "temp_c", "m_temp_c", "temperature (°C)", BYW),
    compare(da, "humidity_pct", "m_humidity_pct", "humidity (%)", BYW),
    compare(da, "wind_speed", "m_wind_speed", "wind speed", BYW),
]).orderBy("variable", "site", "lead_bucket")
display(wx_cmp)

✅ **What you should see:** temperature `corr` 0.9+ and MAE ~1–2 °C. Humidity is usually less accurate. Wind has rows only for pv2 (the only plant with a wind sensor). If the wind `bias` is large and steady, the sensor height or units may differ from Reuniwatt's.

## Step 8 · Which input tracks the target best?
Daytime correlation of each input with the **target** (PV1 actual MW, PV2 `Pv2_AVL`). Reuniwatt uses its **latest** run per horizon (shortest lead) and, for day-ahead, the realistic 12–24 h lead.
- **Measured** sunlight shows the best case: what a perfect weather forecast could reach.
- **Reuniwatt** sunlight and power show what we actually have when forecasting.

In [0]:
latest = fc.withColumn("_rk", F.row_number().over(Window.partitionBy("site", "horizon", "period_end").orderBy("lead_min"))).where("_rk = 1")
wide = latest.groupBy("site", "period_end").pivot("horizon", ["intraday", "dayahead", "weekahead"]).agg(
    F.first("ghi").alias("ghi"), F.first("gti_poa").alias("gti"), F.first("power_mw").alias("power"))
da_1224 = fc.where("horizon = 'dayahead' AND lead_bucket = '12-24h'").groupBy("site", "period_end").agg(
    F.avg("ghi").alias("da1224_ghi"), F.avg("gti_poa").alias("da1224_gti"), F.avg("power_mw").alias("da1224_power"))
t = sens.where("sun_elevation > 5 AND target_mw IS NOT NULL").join(wide, ["site", "period_end"], "left").join(da_1224, ["site", "period_end"], "left")

INPUTS = {
    "measured GHI": "m_ghi", "measured GTI (panels)": "m_gti_poa",
    "Reuniwatt intraday GHI (latest)": "intraday_ghi", "Reuniwatt intraday GTI (latest)": "intraday_gti", "Reuniwatt intraday power (latest)": "intraday_power",
    "Reuniwatt day-ahead GHI (12-24h)": "da1224_ghi", "Reuniwatt day-ahead GTI (12-24h)": "da1224_gti", "Reuniwatt day-ahead power (12-24h)": "da1224_power",
    "Reuniwatt week-ahead power (latest)": "weekahead_power",
}
target_corr = reduce(DataFrame.unionByName, [
    t.groupBy("site").agg(F.count(F.when(F.col(col).isNotNull(), 1)).alias("n"),
                          F.round(F.corr(col, "target_mw"), 3).alias("corr_with_target"),
                          F.round(F.corr(col, "actual_mw"), 3).alias("corr_with_actual_mw")).withColumn("input", F.lit(name))
    for name, col in INPUTS.items()
]).orderBy("site", F.desc("corr_with_target"))
display(target_corr)

✅ **What you should see:** measured panel sunlight (GTI) at or near the top (~0.95+), then Reuniwatt intraday, then day-ahead. Reuniwatt **power** may beat Reuniwatt sunlight, because it already includes the plant's behaviour.
The gap between "measured" and "Reuniwatt day-ahead" is the room for improvement that better weather inputs could give. For pv2, compare `corr_with_target` (available power) with `corr_with_actual_mw`: a lower value for actual MW means curtailment and outages.

## Step 9 · When is Reuniwatt biased? (day-ahead GHI, 12–24 h lead)
Bias by **Gulf hour of day** and by **month**. A steady pattern (e.g. too high every morning) is something a model can learn and correct.

In [0]:
da1224 = cmp.where("horizon = 'dayahead' AND lead_bucket = '12-24h'")
bias_hour = da1224.groupBy("site", F.hour(utc_to_gulf("period_end")).alias("gulf_hour")).agg(
    F.round(F.avg(F.col("ghi") - F.col("m_ghi")), 1).alias("ghi_bias"), F.round(F.avg(F.abs(F.col("ghi") - F.col("m_ghi"))), 1).alias("ghi_mae"),
    F.round(F.avg(F.col("gti_poa") - F.col("m_gti_poa")), 1).alias("gti_bias")).orderBy("site", "gulf_hour")
bias_month = da1224.groupBy("site", F.date_trunc("month", "period_end").alias("month")).agg(
    F.round(F.avg(F.col("ghi") - F.col("m_ghi")), 1).alias("ghi_bias"), F.round(F.avg(F.abs(F.col("ghi") - F.col("m_ghi"))), 1).alias("ghi_mae"),
    F.round(F.corr("ghi", "m_ghi"), 3).alias("ghi_corr")).orderBy("site", "month")
display(bias_hour)
display(bias_month)

✅ **What you should see:** bias close to 0 at midday, possibly larger near sunrise and sunset (timing and low-sun effects). Large positive bias in some months often means dust or haze that Reuniwatt under-predicts.

## Step 10 · Worst days for Reuniwatt
Daily sunlight energy, forecast (day-ahead 12–24 h) vs measured. Biggest errors first, with the day's **measured clear-sky index** (measured ÷ clear-sky: low = cloudy or dusty).

In [0]:
daily = da1224.groupBy("site", F.to_date(utc_to_gulf("period_end")).alias("gulf_day")).agg(
    F.sum("ghi").alias("fc"), F.sum("m_ghi").alias("meas"), F.sum("clearsky_ghi").alias("cs"))
worst = (daily.where("meas > 0")
         .withColumn("error_pct", F.round(100 * (F.col("fc") - F.col("meas")) / F.col("meas"), 1))
         .withColumn("measured_clearsky_index", F.round(F.col("meas") / F.col("cs"), 2))
         .withColumn("forecast_clearsky_index", F.round(F.col("fc") / F.col("cs"), 2))
         .select("site", "gulf_day", "error_pct", "measured_clearsky_index", "forecast_clearsky_index")
         .orderBy(F.desc(F.abs("error_pct"))))
display(worst.limit(20))

✅ **What you should see:** most days within ±5–10%. Worst days usually have a **low measured clear-sky index** (dust storm or cloud) that Reuniwatt missed. Those are the days that drive MAPE, and where sensor-based intraday features help most.

## Step 11 · Write the results file
Writes all tables above to `results/sensor_vs_reuniwatt.txt` next to this notebook. **Commit and push it** (Git dialog) so Claude can read it.

In [0]:
import os
from datetime import datetime


def as_text(df: DataFrame, title: str, max_rows: int = 200) -> str:
    rows = df.limit(max_rows).collect()

    def fmt(v):
        if v is None:
            return ""
        if isinstance(v, float):
            return f"{v:.4g}"
        if isinstance(v, datetime):
            return v.strftime("%Y-%m-%d %H:%M")
        return str(v)
    return "\n".join([f"### {title} ({len(rows)} rows)", " | ".join(df.columns)] + [" | ".join(fmt(v) for v in r) for r in rows]) + "\n"


parts = [
    f"# Sensors vs Reuniwatt · window {START} → {END} · run {datetime.utcnow():%Y-%m-%d %H:%M} UTC · swap date {swap_day}\n",
    as_text(sens.groupBy("site").agg(F.count("*").alias("periods"), F.round(F.max("target_mw")).alias("target_max"),
                                     *[F.round(100 * F.avg(F.col(k).isNotNull().cast("int")), 1).alias(f"{k}_pct") for k in MEAS]), "sensor coverage"),
    as_text(irr_cmp, "step 6 · sunlight: Reuniwatt vs measured (daytime)"),
    as_text(wx_cmp, "step 7 · weather: Reuniwatt day-ahead vs measured"),
    as_text(target_corr, "step 8 · correlation with target (daytime)"),
    as_text(bias_hour, "step 9 · day-ahead 12-24h bias by Gulf hour"),
    as_text(bias_month, "step 9 · day-ahead 12-24h bias by month"),
    as_text(worst, "step 10 · worst days (day-ahead 12-24h GHI energy)", 30),
]
text = "\n".join(parts)
os.makedirs("results", exist_ok=True)
with open("results/sensor_vs_reuniwatt.txt", "w") as f:
    f.write(text)
print(f"Saved {os.path.abspath('results/sensor_vs_reuniwatt.txt')}. Commit and push it.\n")
print(text)